# Integrated Gradients

In [ ]:
# loading in the model
import torch
import os
import random
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from PIL import Image

import torch.nn.functional as F
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet18(weights=None)  # avoids any download
model.fc = nn.Linear(model.fc.in_features, 2)

state = torch.load("best_model.pth", map_location=DEVICE)
model.load_state_dict(state)
model.to(DEVICE).eval()

### Loading in the data
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 32  # keep consistent


def load_npz_split(split):
    data = np.load(f"data/preprocessed/{split}.npz")
    images = torch.from_numpy(data["images"]).float()
    labels = torch.from_numpy(data["labels"]).long()
    return TensorDataset(images, labels)


train_dataset = load_npz_split("train")
val_dataset = load_npz_split("val")
test_dataset = load_npz_split("test")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Baseline choice

Inputs are ImageNet-normalized, so a zero tensor in the model's input space is **not** a black image — it corresponds to the ImageNet mean colour (mid-grey). For chest X-rays, the more principled baseline is a **true black image** (pixel value 0), which after normalization becomes `(0 − mean) / std`. In X-ray physics, black ≈ no attenuation ≈ air, i.e. genuine absence of signal, so attributions read as "what does the model gain over a blank exposure". Alternatives are Gaussian noise (averaged across draws) or a blurred input; we use the black-image baseline as default.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def black_baseline_like(x):
    """Black pixel image (zeros in raw pixel space) mapped to normalized space."""
    return (torch.zeros_like(x) - IMAGENET_MEAN) / IMAGENET_STD


@torch.no_grad()
def denormalize(x):
    return (x * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)


def integrated_gradients(model, x, target, baseline=None, steps=16):
    """Riemann-midpoint Integrated Gradients for a single image.

    Args:
        x:        (1, C, H, W) input in normalized space
        target:   target class index (int)
        baseline: (1, C, H, W) tensor; defaults to black-image baseline
        steps:    number of Riemann steps (default 16)

    Returns:
        attribution: (1, C, H, W) tensor on CPU
        delta:       completeness gap |sum(IG) − (f(x) − f(baseline))|
    """
    model.eval()
    x = x.to(DEVICE)
    if baseline is None:
        baseline = black_baseline_like(x)
    baseline = baseline.to(DEVICE)

    # Midpoint rule: alphas = (k + 0.5) / steps,  k = 0..steps-1
    alphas = (torch.arange(steps, device=DEVICE).float() + 0.5) / steps
    interp = baseline + alphas.view(-1, 1, 1, 1) * (x - baseline)   # (S, C, H, W)
    interp.requires_grad_(True)

    logits = model(interp)
    score = logits[:, target].sum()                                 # sum → per-sample grad
    grads = torch.autograd.grad(score, interp)[0]                   # (S, C, H, W)

    attribution = (x - baseline) * grads.mean(dim=0, keepdim=True)

    with torch.no_grad():
        f_x = model(x)[0, target]
        f_b = model(baseline)[0, target]
        delta = abs(attribution.sum().item() - (f_x - f_b).item())

    return attribution.detach().cpu(), delta

In [ ]:
CLASSES = ["NORMAL", "PNEUMONIA"]


def show_ig(image, attribution, label_idx, pred_idx, title=""):
    img  = denormalize(image.to(DEVICE))[0].cpu().permute(1, 2, 0).numpy()
    attr = attribution[0].sum(dim=0).numpy()                    # collapse channels
    vmax = np.percentile(np.abs(attr), 99) + 1e-12

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(img)
    axes[0].set_title(f"input  (label={CLASSES[label_idx]}, pred={CLASSES[pred_idx]})")
    axes[1].imshow(attr, cmap="seismic", vmin=-vmax, vmax=vmax)
    axes[1].set_title("IG attribution (signed, summed over channels)")
    axes[2].imshow(img)
    axes[2].imshow(attr, cmap="seismic", vmin=-vmax, vmax=vmax, alpha=0.5)
    axes[2].set_title("overlay")
    for ax in axes: ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# Run IG on one example per class from the test set, target = predicted class
shown = {0: False, 1: False}
for img, lbl in test_dataset:
    lbl_i = int(lbl)
    if shown[lbl_i]:
        continue
    shown[lbl_i] = True

    x = img.unsqueeze(0)
    with torch.no_grad():
        pred_i = int(model(x.to(DEVICE)).argmax(dim=1))

    attr, delta = integrated_gradients(model, x, target=pred_i, steps=16)
    show_ig(
        x, attr, lbl_i, pred_i,
        title=f"IG · black baseline · 16 Riemann steps · completeness Δ={delta:.3e}",
    )

    if all(shown.values()):
        break